# NADOR quickstart: monitoring and experiments

NADOR adds monitoring to a 5G network deployed with IFRAN and records experiments on it.
This notebook shows every part of it on one small network (one AMF, SMF, UPF and gNB):

1. Install and import
2. The monitoring profile: metrics, options, images, tuning, and the checks NADOR runs
3. Plan the deployment before installing anything
4. Deploy monitoring, then attach UEs
5. Run an experiment that saves everything
6. Inspect what was saved
7. Collect your own evidence with a custom collector
8. Start and stop probes on a running deployment
9. Check that metrics actually arrive
10. Read the monitoring components' logs
11. Save and restore the deployment handle
12. Remove monitoring

Download this notebook, [monitoring.yaml](monitoring.yaml) and [queries.json](queries.json)
into one folder and use a Python 3.12+ kernel.

Sections 1 and 2 run anywhere. From section 3 on you need the `ran` manager of a deployed
core and RAN, with current inventory and SSH access (see the [RFsim examples](../monitoring/rfsim)
for a full network setup). Keep UEs unattached until monitoring is deployed.

## 1. Install and import

Install once per Python environment, then restart the kernel.

In [ ]:
%pip install --upgrade "git+https://github.com/ifran-project/nador.git@main"

In [ ]:
import asyncio
import csv
import gzip
import json
import subprocess
from datetime import UTC, datetime
from pathlib import Path

from nador import Metrics, MonitoringDeployment, MonitoringManagement, configure_ran_kpis
from nador.artifacts import Collector, ExperimentRun, prometheus_connection
from nador.config import validate_deployment

## 2. The monitoring profile

[monitoring.yaml](monitoring.yaml) says what to collect and where:

- `metrics:` lists **every** metric NADOR knows. `true` turns it on, `false` leaves it out, and a
  mapping turns it on with options. Listing all of them means a profile always says what it collects.
- `targets:` are the network functions to watch. A `role` (`amf`, `smf`, `upf`, `gnb`) marks where
  packets are captured; `container` and `interface` say which container and interface.
- `images:` (optional) overrides container images. Every default is pinned in NADOR's `images.json`.
- `tuning:` (optional) holds rarely changed knobs such as probe intervals.

Namespaces and pod selectors come from `ran`, so the profile needs no credentials.

In [ ]:
config = MonitoringDeployment.from_yaml_file("monitoring.yaml")

for name in Metrics.model_fields:
    state = "on" if config.metrics.running(name) else "off"
    print(f"{name:22} {state}")

### Metric options

Some metrics take options. Probes managed by the controller also accept `on_demand: true`:
they are deployed stopped and can be started later (section 8). In YAML this reads:

```yaml
metrics:
  rtt: {packet_pairing: false, slices: ["01:ffffff"]}  # only measure slice 01:ffffff
  logs: true
  ngap: true                                           # AMF session-setup metrics
  pfcp: true                                           # SMF session-establishment metrics
  scheduler_wait: {on_demand: true}                    # CPU scheduler probe, started later
tuning:
  probe_interval_seconds: 10
```

The cell below builds the same variant from the loaded profile, so you can try options
without editing the file. `validate_deployment` applies the same checks as reading a file.

In [ ]:
data = config.model_dump()
data["metrics"] |= {
    "rtt": {"packet_pairing": False, "slices": ["01:ffffff"]},
    "logs": True,
    "ngap": True,
    "pfcp": True,
    "scheduler_wait": {"on_demand": True},
}
data["tuning"] |= {"probe_interval_seconds": 10}
richer = validate_deployment(data)

print("running:  ", [name for name in Metrics.model_fields if richer.metrics.running(name)])
print("on demand:", [name for name in Metrics.model_fields if richer.metrics.available(name) and not richer.metrics.running(name)])
print("rtt slices:", richer.metrics.rtt.slices)
print("host probe images:", richer.host_probe_images())

### Images

Every image NADOR deploys has a key in `images.json`. Override one by key, for example a
custom RTT probe build. Images must be pinned to a tag or digest.

In [ ]:
print("default RTT probe:", config.image("rtt_probe"))

data = config.model_dump()
data["images"] = {"rtt_probe": "ghcr.io/your-org/ebpf-latency-probe:v2"}
custom = validate_deployment(data)
print("custom RTT probe: ", custom.image("rtt_probe"))

### Mistakes are caught before anything is deployed

NADOR checks that each metric has what it needs. Here `upf_transit` is turned on without its
test peer, and a probe image has no tag:

In [ ]:
data = config.model_dump()
data["metrics"] |= {"upf_transit": True}
data["images"] = {"controller": "ghcr.io/your-org/nador-controller"}

try:
    validate_deployment(data)
except ValueError as error:
    print(error)

Pick the profile to deploy. This notebook uses the file as is; use `richer` to deploy the
variant built above.

In [ ]:
profile = config  # or: richer

## 3. Plan the deployment

Run your network setup first so that `ran` exists. `plan()` locates every target in the deployed
network and renders all Helm values **without touching the cluster**, so you can check what will be
installed.

In [ ]:
monitoring = MonitoringManagement(preceding=ran, config=profile)
plan = monitoring.plan()

print("control node:", plan.control_node)
for target in plan.targets:
    print(f"target {target.component:6} role={target.role} namespace={target.namespace} {target.label_selectors}")
print("Helm releases:", list(plan.components))

**OCUDU RANs only.** To collect OCUDU's native KPIs, turn on `ran_kpis` and enable the RAN's JSON
metrics **before** deploying the RAN:

```python
configure_ran_kpis(ran, profile)  # then: await ran.deploy()
```

OAI RANs need no extra step.

## 4. Deploy monitoring, then attach UEs

`deploy()` installs Prometheus, Grafana (`admin` / `admin`), the probes and, when a metric needs
UE context, the UE mapper and the AMF/SMF sniffers. Deploy before attaching UEs, so the sniffers
see session establishment.

In [ ]:
await monitoring.deploy()

Then attach the UEs. For containerized UEs managed by IFRAN the usual sequence is below; physical
UEs connect with their normal procedure.

In [ ]:
from ifran.deployment import UEManagement

ue = UEManagement(preceding=ran)
await ue.prepare()
await ue.install()

## 5. Run an experiment

`ExperimentRun` records when each `section` of your workload ran and, with the monitoring manager,
saves everything the deployment observes meanwhile:

- metrics from Prometheus, for the queries in `queries.json`, split per section;
- logs from Loki, when `logs` is on;
- the UE context over time (UE, IP, slice, tunnels), when the UE mapper is deployed;
- snapshots of the hosts and the cluster before and after the run.

Choose a UE component, its tunnel interface and an address reachable through its 5G connection.
The UE container must provide `ping`.

In [ ]:
UE_COMPONENT = "sim_ue"
UE_INTERFACE = "oaitun_ue1"
TARGET_IP = "<address reachable through the UE tunnel>"

result_dir = Path("results") / f"{datetime.now(UTC):%Y%m%dT%H%M%S}-ping"

async with ExperimentRun(result_dir, name="ue-ping", monitoring=monitoring, queries_file="queries.json") as run:
    with run.section("idle"):
        await asyncio.sleep(5)  # a baseline to compare against

    with run.section("ping"):
        report = await ue.run_command(
            command=["ping", "-I", UE_INTERFACE, "-c", "10", TARGET_IP],
            components=[UE_COMPONENT],
            quiet=True,
            flatten=False,
        )
    run.save_json("ping.json", report)  # your own results go in the same folder

    await asyncio.sleep(2)  # let Prometheus scrape the last samples

print(f"Saved to {result_dir.resolve()}")

If a section raises, the run is recorded as `failed` and the exception reaches you as usual;
evidence is still collected. If collection itself has a problem on a successful run, the run raises
`RuntimeError` and `experiment_metadata.json` lists what went wrong.

Options tune what is collected, for example `collect_setup=False` to skip host snapshots,
`interval_seconds=1` to sample the UE context more often, or `loki_selector='{namespace="core"}'`
to export fewer logs.

## 6. Inspect what was saved

In [ ]:
for path in sorted(result_dir.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size:>10}  {path.relative_to(result_dir)}")

- `experiment_metadata.json`: status (`completed`, `failed` or `cancelled`) and any collection errors.
- `timeline_summary.csv`, `timeline_recap.txt`: when the campaign and each section ran.
- `prometheus_timeseries.csv`: every sample of every query; `prometheus_export_summary.json` says
  how many rows each query returned (an empty query had no samples, which is not an error).
- `by_window/section/<name>/`: the same samples cut to one section.
- `ue_context/`: sampled UE inventory; `setup/before|after/`: host and cluster snapshots;
  `logs/`: Loki export, when logs are on.

In [ ]:
metadata = json.loads((result_dir / "experiment_metadata.json").read_text())
print("status:", metadata["status"], "| collection errors:", metadata["collection_errors"])

print((result_dir / "timeline_recap.txt").read_text())

with gzip.open(result_dir / "by_window/section/section__ping/prometheus_timeseries.csv.gz", "rt") as f:
    rows = list(csv.DictReader(f))
print(f"{len(rows)} samples during 'ping', queries: {sorted({row['query_name'] for row in rows})}")

## 7. Collect your own evidence

A collector is any object with up to three steps, all optional:

- `start(run)`: before the workload,
- `stop(run)`: after the workload (in reverse order),
- `collect(run, start, end)`: after the timeline is written, for the run's time range.

If a step fails, the error is recorded and the other collectors still run. This one saves the UE's
addresses before and after the workload:

In [ ]:
class UeAddresses(Collector):
    label = "ue_addresses"

    async def _save(self, run, phase):
        output = await ue.run_command(["ip", "-j", "address"], components=[UE_COMPONENT], quiet=True, flatten=False)
        run.write_json(f"ue_addresses/{phase}.json", output)

    async def start(self, run):
        await self._save(run, "before")

    async def stop(self, run):
        await self._save(run, "after")


result_dir = Path("results") / f"{datetime.now(UTC):%Y%m%dT%H%M%S}-custom"
async with ExperimentRun(result_dir, monitoring=monitoring, collectors=[UeAddresses()]) as run:
    with run.section("wait"):
        await asyncio.sleep(5)

print(sorted(path.name for path in (result_dir / "ue_addresses").iterdir()))

## 8. Start and stop probes on a running deployment

Metrics deployed with `on_demand: true` can be started later without redeploying. The
`nador-controller` command talks to the controller's API; forward its port from the control node
first (it only accepts plain HTTP on localhost):

```bash
ssh -L 8787:localhost:8787 <control-node> \
    kubectl -n monitoring port-forward deploy/monitoring-controller 8787:8787
nador-controller list
nador-controller start rtt --slice 01:ffffff
nador-controller stop rtt
```

One-way latency metrics are switched from Python:

```python
await monitoring.set_running(n3_one_way=True)   # needs metrics.n3_one_way in the profile
await monitoring.set_running(n3_one_way=False)
```

## 9. Check that metrics actually arrive

A finished experiment does not prove every metric had data. `nador-check-metrics` runs every
query against Prometheus and reports which ones returned samples, and which scrape targets are down.

In [ ]:
report = Path("results") / "metrics-check.json"
async with prometheus_connection(monitoring) as url:
    subprocess.run(["nador-check-metrics", "--prometheus-url", url, "--out", str(report)], check=False)
print(json.loads(report.read_text())["counts"])

## 10. Read the monitoring components' logs

In [ ]:
print(await monitoring.get_logs(components=["ue-mapper-api"], since_seconds=600, quiet=True))

## 11. Save and restore the deployment handle

Save the models after deploying. After a kernel restart, restore the handle without redeploying.
Keep these files private: they contain host access details.

In [ ]:
monitoring.output(metamodel_file="monitoring_metamodel.yaml", deployment_model_file="monitoring_deployment_model.yaml")

# After a restart:
monitoring = MonitoringManagement.from_yaml_files("monitoring_metamodel.yaml", "monitoring_deployment_model.yaml")

## 12. Remove monitoring

This removes monitoring only; the core, RAN, UEs and saved results stay.

In [ ]:
# await monitoring.remove()